# Target-caption diffusion & concept/token distribution — v1

**Experiment:** 10 aligned `(source, target)` pairs from the [Zintom adversarial dataset](https://www.kaggle.com/datasets/zintom/blackbox-adversarial-attack-dataset); Stable Diffusion v1.5 image-to-image generation conditioned on the **original target caption**; diagnostic attention/gradient/region analysis across CLIP ViT-B/16, CLIP ViT-B/32, **OpenCLIP ViT-g/14** (LAION), BLIP-2, and Qwen3-VL. The generated image is an **experimental target**, not an adversarial sample. This notebook does not estimate attack success rate.

### Research protocol / avoiding misinterpretation

- Diffusion baseline mirrors the supplied `kltn-attack-v5.1-diffusion_regions(4).ipynb`: Stable Diffusion v1.5, `img2img(source, target_caption)`, 40 denoising steps, `strength=.7`, CFG=7.5. Investigate caption length, `strength`, and seed explicitly.
- **CLIP image self-attention is caption independent.** By contrast the input-image gradient of image–text cosine *depends on the supplied caption/concept*. Do not label an attention-only heatmap “concept grounding”. CLIP text-token gradient is a sensitivity diagnostic, not causal evidence.
- BLIP-2 Q-Former **cross-attention** is from learned query tokens to vision tokens; **it is not directly conditioned on caption words** for the OPT variant. Its text-conditioned gradient uses language loss and is logged separately.
- Qwen3-VL text-query-to-image-token attention is computed at **one selected text query token**, and pixel-patch gradient from a yes/no logit contrast. This is an exploratory diagnostic, **not a faithful causal explanation**.
- Region boxes come from thresholding the measured heatmaps, **not from an object detector / grounding model**. The FINER-inspired caption concepts are strictly checked against the caption to suppress hallucinations. BiCo supplies the methodological idea of binding concepts to prompt tokens; pretrained BiCo adapters are **not** used here.
- All models are run **sequentially**: diffusion and CLIP can use one GPU; optional 4-bit BLIP-2 and Qwen3-VL can spread across two T4s. Intermediate arrays and tables are persisted before models are unloaded.
- Diffusion safety-filtered outputs are recorded as rejected and **not replaced silently** by original targets. Rerunning cells resumes from cached files. Source and target captions are kept frozen within a run.

**References (code):** [FINER object prompts](https://github.com/ExplainableML/finer/blob/main/train/generate_training_data/prompts/object_prompts.py), [BiCo prompt augmentation](https://github.com/refkxh/BiCo/blob/main/bico/prompt_aug.py), [MPCAttack code organization](https://github.com/LiYuanBoJNU/MPCAttack/blob/main/generate_adversarial_examples_MPCAttack.py), [Diffusers img2img](https://huggingface.co/docs/diffusers/using-diffusers/img2img), [OpenCLIP checkpoints](https://github.com/mlfoundations/open_clip/blob/main/docs/PRETRAINED.md).

> **Hardware:** Kaggle GPU ×2, Internet ON for initial model downloads; Kaggle Add Input dataset. Start with the configuration smoke setting `N_SAMPLES=1` to check your environment; then set back to `10`. Large models can fail because of disk/VRAM/quantization incompatibilities; explicit status logs are exported, not converted to fake maps.

In [ ]:
# 00 — Dependencies. Execute once on a fresh Kaggle session, then restart if requested.
# Do NOT reinstall torch/torchvision (keep Kaggle CUDA-compatible wheels).
%pip -q install -U "transformers>=4.57,<5" "diffusers>=0.34,<0.39" \
    "accelerate>=1.3" "bitsandbytes>=0.45" "open_clip_torch>=3.0" \
    "safetensors>=0.4" "sentencepiece" "protobuf" "ftfy" "einops" "opencv-python-headless" \
    "pandas>=2" "matplotlib" "scipy" "tqdm"

In [ ]:
# 01 — Single source of truth (edit this cell)
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Tuple

@dataclass(frozen=True)
class Config:
    DATASET_ROOT: Path = Path('/kaggle/input/blackbox-adversarial-attack-dataset')
    SOURCE_DIR: Optional[Path] = None
    TARGET_DIR: Optional[Path] = None
    PAIR_MANIFEST: Optional[Path] = None  # optional CSV: source,target, with relative/absolute paths
    OUTPUT_DIR: Path = Path('/kaggle/working/analysis_diffusion_concept_v1')
    N_SAMPLES: int = 10
    SAMPLE_START: int = 0
    SEED: int = 2026

    # Caption comes from BLIP image captioner. No ground-truth caption is invented.
    CAPTION_ID: str = 'Salesforce/blip-image-captioning-base'
    CAPTION_BACKEND: str = 'blip'  # or 'qwen3vl': optional longer, image-grounded captions
    QWEN_CAPTION_PROMPT: str = 'Describe this image faithfully in about 40 to 60 English words. Include visible objects, attributes, actions and layout. Do not speculate.'
    CAPTION_MAX_NEW_TOKENS: int = 120
    CONCEPT_LM_ID: str = 'Qwen/Qwen2.5-1.5B-Instruct'
    CONCEPT_MAX_NEW_TOKENS: int = 90
    MAX_CONCEPTS: int = 4
    USE_LM_CONCEPT_EXTRACTOR: bool = True

    RUN_DIFFUSION: bool = True
    DIFFUSION_ID: str = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
    DIFFUSION_STEPS: int = 40
    DIFFUSION_CFG: float = 7.5
    DIFFUSION_STRENGTHS: Tuple[float,...] = (0.7,)
    CAPTION_WORD_BUDGETS: Tuple[int,...] = (12, 24, 40)
    DIFFUSION_SEEDS: Tuple[int,...] = (2026,)
    DIFFUSION_IMAGE_SIDE: int = 512
    DIFFUSION_INIT_IMAGE: str = 'source'  # 'source' or 'target'; original protocol: 'source'

    RUN_CLIP: bool = True
    CLIP_MODELS: Tuple[str,...] = ('clip_b16', 'clip_b32', 'clip_g14')
    CLIP_MAX_CONCEPTS: int = 3
    RUN_BLIP2: bool = True
    BLIP2_ID: str = 'Salesforce/blip2-opt-2.7b'
    BLIP2_ENABLE_GRAD: bool = True
    RUN_QWEN: bool = True
    QWEN_ID: str = 'Qwen/Qwen3-VL-8B-Instruct'
    QWEN_ENABLE_GRAD: bool = True
    QWEN_MAX_PIXELS: int = 196 * 28 * 28   # limit visual tokens for T4
    QWEN_MAX_CONCEPTS: int = 2

    HEATMAP_QUANTILE: float = 0.80
    MAX_BOXES: int = 3
    STRICT_MODELS: bool = False  # True = raise when a requested model fails

CFG = Config()
assert CFG.N_SAMPLES > 0
assert CFG.CAPTION_BACKEND in {'blip','qwen3vl'}
assert CFG.DIFFUSION_INIT_IMAGE in {'source', 'target'}
assert all(0.0 < s <= 1.0 for s in CFG.DIFFUSION_STRENGTHS)
assert all(0 < x <= 75 for x in CFG.CAPTION_WORD_BUDGETS)
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(CFG)

In [ ]:
# 02 — Imports / runtime checks / output lifecycle
import contextlib, gc, hashlib, json, math, os, random, re, time, traceback, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from tqdm.auto import tqdm
import torch
import torch.nn.functional as F

random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
torch.manual_seed(CFG.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.SEED)
print('Torch:', torch.__version__)
print('CUDA GPUs:', torch.cuda.device_count())
for k in range(torch.cuda.device_count()):
    print(' ', k, torch.cuda.get_device_name(k), 'VRAM GB:', round(torch.cuda.get_device_properties(k).total_memory/2**30, 2))
if not torch.cuda.is_available():
    raise EnvironmentError('This notebook requires Kaggle GPU runtime; choose GPU T4 x2.')

GPU0 = 'cuda:0'
GPU1 = 'cuda:1' if torch.cuda.device_count() > 1 else GPU0
ARTIFACTS = CFG.OUTPUT_DIR / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
RUN_STATUS = []

def save_json(path, data):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False, default=str), encoding='utf-8')

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def release(*objects):
    for obj in objects: del obj
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def first_device(model):
    return next(p.device for p in model.parameters())

def model_error(model, sample_id, exc):
    message = f'{type(exc).__name__}: {str(exc)[:350]}'
    RUN_STATUS.append({'model': model, 'sample_id': sample_id, 'status': 'error', 'detail': message})
    print(f'[ERROR] {model} sample={sample_id}: {message}')
    if CFG.STRICT_MODELS: raise exc

def output_image(path):
    with Image.open(path) as im: return im.convert('RGB').copy()

def square_preview(img, side=224):
    from PIL import ImageOps
    return ImageOps.fit(img.convert('RGB'), (side,side), method=Image.Resampling.BICUBIC)

## A. Dataset, pairing and frozen target captions

The older attack notebook expects source images inside **`bigscale/`** and target images inside **`target_images/`**, normally with class subdirectories. We enforce identifier matching when filenames use IDs (`00000_source`, `00000_target`). If dataset filenames differ, provide an explicit CSV manifest; **never silently zip unrelated sorted files**.

In [ ]:
# 03 — Dataset pair index (no network call; expects Kaggle Add Input)
IMAGE_EXT = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}

def find_role_dir(root: Path, folder_name: str, override=None) -> Path:
    if override is not None:
        found = Path(override)
        if not found.is_dir(): raise FileNotFoundError(found)
        return found
    candidates = [root, Path('/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset'),
                  Path('/kaggle/input/blackbox-adversarial-attack-dataset')]
    matches = sorted({str(p) for base in candidates if base.exists()
                      for p in ([base] if base.name == folder_name else []) + list(base.rglob(folder_name))
                      if p.is_dir()})
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected ONE {folder_name} directory, found {matches}. Set SOURCE_DIR/TARGET_DIR.')
    return Path(matches[0])

def list_images(folder: Path):
    return sorted(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXT)

def sample_key(path: Path):
    match = re.match(r'^(\d{3,})', path.stem)
    return match.group(1) if match else None

def read_pair_manifest(path: Path):
    frame = pd.read_csv(path)
    if not {'source','target'} <= set(frame.columns):
        raise ValueError('Manifest must include source,target columns')
    result = []
    for i, row in frame.iterrows():
        src, tgt = [Path(str(row[k])) for k in ('source','target')]
        src = src if src.is_absolute() else path.parent / src
        tgt = tgt if tgt.is_absolute() else path.parent / tgt
        result.append((src,tgt))
    return result

def discover_pairs(cfg):
    if cfg.PAIR_MANIFEST:
        raw = read_pair_manifest(cfg.PAIR_MANIFEST)
    else:
        a = list_images(find_role_dir(cfg.DATASET_ROOT,'bigscale',cfg.SOURCE_DIR))
        b = list_images(find_role_dir(cfg.DATASET_ROOT,'target_images',cfg.TARGET_DIR))
        if not a or not b: raise ValueError('Dataset contains no supported images')
        src_ids, tgt_ids = [sample_key(p) for p in a], [sample_key(p) for p in b]
        if all(src_ids) and all(tgt_ids):
            if len(set(src_ids)) != len(src_ids) or len(set(tgt_ids)) != len(tgt_ids):
                raise ValueError('Duplicate sample ID prefix: create explicit manifest')
            source_map, target_map = dict(zip(src_ids,a)), dict(zip(tgt_ids,b))
            if source_map.keys() != target_map.keys():
                raise ValueError('Source and target IDs differ. Create explicit manifest; no index pairing.')
            raw = [(source_map[i], target_map[i]) for i in sorted(source_map)]
        else:
            raise ValueError('Cannot verify pair IDs from filenames; supply PAIR_MANIFEST CSV')
    raw = raw[cfg.SAMPLE_START:cfg.SAMPLE_START+cfg.N_SAMPLES]
    if len(raw) != cfg.N_SAMPLES: raise ValueError(f'Only {len(raw)} pairs found; requested {cfg.N_SAMPLES}')
    result = []
    for idx, (a,b) in enumerate(raw, start=cfg.SAMPLE_START):
        if not a.is_file() or not b.is_file(): raise FileNotFoundError((a,b))
        result.append({'sample_id':sample_key(a) or f'{idx:05d}',
                       'index':idx,'source':str(a),'target':str(b),
                       'source_sha256':sha256(a), 'target_sha256':sha256(b)})
    if len({r['sample_id'] for r in result}) != len(result): raise ValueError('Duplicate sample IDs')
    return result

PAIRS = discover_pairs(CFG)
save_json(ARTIFACTS/'pairs.json', PAIRS)
display(pd.DataFrame(PAIRS)[['sample_id','source','target']])

In [ ]:
# 04 — Visual verification before generating any captions/targets

def show_original_pairs(pairs, limit=5):
    show = pairs[:limit]
    fig, axes = plt.subplots(len(show), 2, figsize=(9, 3.4*len(show)), squeeze=False)
    for i, pair in enumerate(show):
        for j, role in enumerate(('source','target')):
            axes[i,j].imshow(output_image(pair[role]))
            axes[i,j].set_title(f'{pair["sample_id"]}: {role}'); axes[i,j].axis('off')
    plt.tight_layout(); plt.show()

show_original_pairs(PAIRS)

In [ ]:
# 05 — Model-generated SOURCE and TARGET captions; BLIP light / Qwen3VL detailed
from transformers import BlipProcessor, BlipForConditionalGeneration

CAPTION_CACHE = ARTIFACTS/'captions.json'

def load_caption_backend(cfg):
    if cfg.CAPTION_BACKEND == 'blip':
        processor=BlipProcessor.from_pretrained(cfg.CAPTION_ID)
        model=BlipForConditionalGeneration.from_pretrained(cfg.CAPTION_ID,
                    torch_dtype=torch.float16).to(GPU1).eval()
    else:
        from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig
        processor=AutoProcessor.from_pretrained(cfg.QWEN_ID,
                    min_pixels=64*28*28,max_pixels=cfg.QWEN_MAX_PIXELS)
        quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
                    bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
        model=Qwen3VLForConditionalGeneration.from_pretrained(cfg.QWEN_ID,
                    quantization_config=quant,device_map='auto',torch_dtype=torch.float16).eval()
    model.requires_grad_(False)
    return model,processor

def caption_image(model,processor,path,cfg):
    image=output_image(path)
    if cfg.CAPTION_BACKEND == 'blip':
        batch=processor(images=image,return_tensors='pt').to(first_device(model))
        with torch.inference_mode():
            ids=model.generate(**batch,max_new_tokens=cfg.CAPTION_MAX_NEW_TOKENS,num_beams=3)
        return processor.batch_decode(ids,skip_special_tokens=True)[0].strip()
    messages=[{'role':'user','content':[{'type':'image','image':image},
               {'type':'text','text':cfg.QWEN_CAPTION_PROMPT}]}]
    batch=processor.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,
               return_dict=True,return_tensors='pt')
    entry=first_device(model)
    batch={k:v.to(entry) if isinstance(v,torch.Tensor) else v for k,v in batch.items()}
    with torch.inference_mode():
        ids=model.generate(**batch,max_new_tokens=cfg.CAPTION_MAX_NEW_TOKENS,
                           do_sample=False,use_cache=True)
    return processor.batch_decode(ids[:,batch['input_ids'].shape[1]:],
                                  skip_special_tokens=True)[0].strip()

def create_captions(pairs,cfg):
    old=json.loads(CAPTION_CACHE.read_text()) if CAPTION_CACHE.is_file() else {}
    model=processor=None
    try:
        for pair in tqdm(pairs,desc=f'{cfg.CAPTION_BACKEND} captions'):
            sid=pair['sample_id']
            identity={'source':pair['source_sha256'],'target':pair['target_sha256'],
                      'caption_model':cfg.CAPTION_ID if cfg.CAPTION_BACKEND=='blip' else cfg.QWEN_ID,
                      'caption_backend':cfg.CAPTION_BACKEND,
                      'qwen_prompt':cfg.QWEN_CAPTION_PROMPT if cfg.CAPTION_BACKEND=='qwen3vl' else None,
                      'max_new_tokens':cfg.CAPTION_MAX_NEW_TOKENS}
            if sid in old and old[sid].get('identity')==identity:continue
            if model is None:model,processor=load_caption_backend(cfg)
            src=caption_image(model,processor,pair['source'],cfg)
            tgt=caption_image(model,processor,pair['target'],cfg)
            if not src or not tgt:raise ValueError(f'Empty caption for {sid}')
            old[sid]={'identity':identity,'source_caption':src,'target_caption':tgt}
            save_json(CAPTION_CACHE,old)
    finally:
        del model,processor
        release()
    return old

CAPTIONS=create_captions(PAIRS,CFG)
display(pd.DataFrame([{'id':p['sample_id'],
                       'source':CAPTIONS[p['sample_id']]['source_caption'],
                       'target':CAPTIONS[p['sample_id']]['target_caption']}
                      for p in PAIRS]))

### Captions → concepts (FINER-like constrained extraction)

FINER extracts up to five concrete objects already present in a caption and avoids inferred objects. Here a lightweight instruction LM proposes structured concepts and a strict phrase-in-caption validator accepts only grounded strings. **No counterfactual objects are generated at this stage.** A transparent lexical fallback keeps the notebook usable when the concept LM cannot be loaded; rows disclose their extraction method. Extraction is **not** the full FINER code or a trained BiCo concept adapter.

In [ ]:
# 06 — Concept extractor; fast exact-span guard, separate from diffusion prompts
from transformers import AutoModelForCausalLM, AutoTokenizer

CONCEPT_CACHE = ARTIFACTS/'concepts.json'
FINER_STYLE_PROMPT = """Extract up to {limit} visually concrete object/concept phrases from this image caption.
Every phrase MUST be a literal contiguous substring of the caption, and must name a visual entity,
attribute-object phrase, or spatial relation actually mentioned. Do not infer or invent things.
Return ONLY a JSON array of short strings. Example: ["red bus", "glass building", "statue"].
Caption: {caption}"""

STOPWORDS = {'the','this','that','there','with','from','under','into','upon','over',
             'while','about','inside','outside','which','where','they','them','are','was',
             'has','have','and','for','near','next','around','very','some','is','of','in','on','a','an'}

def clean_caption_words(text):
    return re.findall(r"[A-Za-z]+(?:[-'][A-Za-z]+)?|\d+", text)

def literal_concept_check(candidate, caption):
    phrase = re.sub(r'\s+', ' ', str(candidate)).strip(' .,:;"\'')
    if len(phrase) < 3 or len(phrase.split()) > 5: return None
    if not re.search(r'(?<!\w)'+re.escape(phrase)+r'(?!\w)',caption,re.I): return None
    if phrase.lower() in STOPWORDS: return None
    return phrase

def parse_concept_json(reply):
    match = re.search(r'\[[\s\S]*?\]', reply)
    if not match: return []
    try:
        arr = json.loads(match.group(0))
        return arr if isinstance(arr,list) else []
    except json.JSONDecodeError: return []

def lexical_fallback(caption, limit):
    tokens = clean_caption_words(caption)
    phrases = []
    for k in (2,1):
        for i in range(len(tokens)-k+1):
            phrase = ' '.join(tokens[i:i+k])
            if tokens[i].lower() in STOPWORDS or tokens[i+k-1].lower() in STOPWORDS: continue
            exact=literal_concept_check(phrase,caption)
            if exact and exact.lower() not in {p.lower() for p in phrases}: phrases.append(exact)
    return phrases[:limit]

def extract_concepts_lm(model, tokenizer, caption, limit):
    prompt = FINER_STYLE_PROMPT.format(limit=limit,caption=caption)
    messages = [{'role':'system','content':'Follow extraction instructions exactly.'},
                {'role':'user','content':prompt}]
    template = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(template, return_tensors='pt').to(first_device(model))
    with torch.inference_mode():
        outputs = model.generate(**inputs,max_new_tokens=CFG.CONCEPT_MAX_NEW_TOKENS,do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(outputs[0,inputs.input_ids.shape[1]:],skip_special_tokens=True)
    return parse_concept_json(text), text

def create_concept_records(pairs, captions, cfg):
    store = json.loads(CONCEPT_CACHE.read_text()) if CONCEPT_CACHE.is_file() else {}
    lm = tok = None
    for pair in tqdm(pairs,desc='Concept extraction'):
        sid = pair['sample_id']; caption = captions[sid]['target_caption']
        identity = {'caption':caption,'model':cfg.CONCEPT_LM_ID if cfg.USE_LM_CONCEPT_EXTRACTOR else 'lexical',
                    'limit':cfg.MAX_CONCEPTS}
        if sid in store and store[sid].get('identity') == identity: continue
        proposals, raw, method = [], '', 'lexical_fallback'
        if cfg.USE_LM_CONCEPT_EXTRACTOR:
            try:
                if lm is None:
                    tok = AutoTokenizer.from_pretrained(cfg.CONCEPT_LM_ID)
                    lm = AutoModelForCausalLM.from_pretrained(cfg.CONCEPT_LM_ID,
                              torch_dtype=torch.float16, low_cpu_mem_usage=True).to(GPU0).eval()
                proposals,raw = extract_concepts_lm(lm,tok,caption,cfg.MAX_CONCEPTS)
                method = 'lm_literal_filtered'
            except Exception as exc:
                model_error('concept_lm',sid,exc)
                method = 'lexical_fallback_after_error'
        accepted = []
        for phrase in proposals:
            valid = literal_concept_check(phrase,caption)
            if valid and valid.lower() not in [q.lower() for q in accepted]: accepted.append(valid)
        if not accepted:
            accepted = lexical_fallback(caption,cfg.MAX_CONCEPTS)
            method += '+fallback'
        if not accepted:
            raise RuntimeError(f'No exact-caption concept could be extracted for {sid}; review the caption')
        store[sid] = {'identity':identity,'concepts':accepted[:cfg.MAX_CONCEPTS],
                      'method':method,'raw_model_output':raw}
        save_json(CONCEPT_CACHE,store)
    del lm,tok
    release()
    return store

CONCEPT_RECORDS = create_concept_records(PAIRS,CAPTIONS,CFG)
display(pd.DataFrame([{'id':r['sample_id'], 'caption':CAPTIONS[r['sample_id']]['target_caption'],
                       'concepts':CONCEPT_RECORDS[r['sample_id']]['concepts'],
                       'method':CONCEPT_RECORDS[r['sample_id']]['method']} for r in PAIRS]))

## B. Caption-length/strength diffusion sweep

**Main comparison:** Source-image img2img, same seed, same source image, and multiple text budgets. Caption budgets truncate whole words (not diffusion tokenizer tokens), so the notebook also logs the exact Stable Diffusion CLIP tokenizer length and whether the prompt was truncated at its 77-token limit. The BLIP caption may be shorter than 24/40 words, so those nominal budget values can map to **the same actual caption**; duplicate effective prompts are cached/deduplicated.

The generated image receives no ground-truth target caption beyond the frozen original target caption. Use source/target/generated contact sheets before interpreting the heatmaps.

In [ ]:
# 07 — Pure helpers: prompt length, resize, hash/cache metadata

def truncate_words(text, budget):
    words = text.split()
    return ' '.join(words[:budget])

def diffusion_input_size(im, side):
    w,h = im.size
    scale = side/max(w,h)
    return (max(64,round(w*scale/8)*8), max(64,round(h*scale/8)*8))

def fit_diffusion(im, side):
    return im.convert('RGB').resize(diffusion_input_size(im,side), Image.Resampling.LANCZOS)

def diffusion_jobs(pairs, captions, cfg):
    jobs = []
    for p in pairs:
        sid = p['sample_id']; text = captions[sid]['target_caption']
        seen = set()
        for budget in cfg.CAPTION_WORD_BUDGETS:
            prompt = truncate_words(text,budget)
            for strength in cfg.DIFFUSION_STRENGTHS:
                for seed in cfg.DIFFUSION_SEEDS:
                    key = (prompt,float(strength),int(seed))
                    if key in seen: continue
                    seen.add(key)
                    job_id = hashlib.sha1(repr((sid,key,cfg.DIFFUSION_INIT_IMAGE)).encode()).hexdigest()[:12]
                    jobs.append({'sample_id':sid,'job_id':job_id,'budget':budget,
                                 'actual_words':len(prompt.split()),'prompt':prompt,
                                 'strength':float(strength),'seed':int(seed),
                                 'source':p['source'],'target':p['target']})
    return jobs

DIFFUSION_JOBS = diffusion_jobs(PAIRS,CAPTIONS,CFG)
print(f'Diffusion jobs: {len(DIFFUSION_JOBS)}; distinct caption lengths:',
      sorted({j['actual_words'] for j in DIFFUSION_JOBS}))
display(pd.DataFrame(DIFFUSION_JOBS)[['sample_id','budget','actual_words','strength','seed']].head(20))

In [ ]:
# 08 — Diffusion loader / reusable generation cell
from diffusers import StableDiffusionImg2ImgPipeline

DIFF_DIR = ARTIFACTS/'diffusion'
DIFF_DIR.mkdir(exist_ok=True)

def load_diffuser(cfg):
    pipe = StableDiffusionImg2ImgPipeline.from_pretrained(cfg.DIFFUSION_ID,
             torch_dtype=torch.float16).to(GPU0)
    pipe.enable_vae_slicing()
    pipe.enable_attention_slicing()
    return pipe

def diffusion_token_stats(pipe,prompt):
    enc = pipe.tokenizer(prompt, add_special_tokens=True, truncation=False)
    count = len(enc['input_ids'])
    return {'sd_token_count':count,'sd_truncated':count>pipe.tokenizer.model_max_length,
            'sd_max_length':pipe.tokenizer.model_max_length}

def render_diffusion(pipe,job,cfg):
    init = output_image(job[cfg.DIFFUSION_INIT_IMAGE])
    init = fit_diffusion(init,cfg.DIFFUSION_IMAGE_SIDE)
    generator = torch.Generator(device=GPU0).manual_seed(job['seed'])
    with torch.inference_mode():
        result = pipe(prompt=job['prompt'],image=init,strength=job['strength'],
                      guidance_scale=cfg.DIFFUSION_CFG,num_inference_steps=cfg.DIFFUSION_STEPS,
                      generator=generator)
    if any(getattr(result,'nsfw_content_detected',None) or []):
        raise RuntimeError('Pipeline rejected/filtered image')
    out = result.images[0].convert('RGB')
    if out.getbbox() is None: raise RuntimeError('Empty all-black output; inspect filtering')
    return out

def run_diffusion(jobs,cfg):
    manifest_path = DIFF_DIR/'manifest.json'
    records = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
    pipe = None
    try:
        for job in tqdm(jobs,desc='Stable Diffusion img2img'):
            cache_id = f'{job["sample_id"]}_{job["job_id"]}'
            image_path = DIFF_DIR/f'{cache_id}.png'
            settings = {'job':job, 'model':cfg.DIFFUSION_ID,'steps':cfg.DIFFUSION_STEPS,
                        'guidance':cfg.DIFFUSION_CFG,'side':cfg.DIFFUSION_IMAGE_SIDE,
                        'init_role':cfg.DIFFUSION_INIT_IMAGE,
                        'source_sha256':sha256(job['source']),'target_sha256':sha256(job['target'])}
            identity = hashlib.sha256(json.dumps(settings,sort_keys=True).encode()).hexdigest()
            if (cache_id in records and records[cache_id].get('identity')==identity and
                records[cache_id].get('status')=='ok' and image_path.exists() and
                sha256(image_path)==records[cache_id]['image_sha256']): continue
            try:
                if pipe is None: pipe = load_diffuser(cfg)
                stats = diffusion_token_stats(pipe,job['prompt'])
                out = render_diffusion(pipe,job,cfg)
                out.save(image_path)
                records[cache_id] = {'status':'ok', 'identity':identity,'image':str(image_path),
                    'image_sha256':sha256(image_path), 'job':job,**stats}
            except Exception as exc:
                records[cache_id] = {'status':'error','identity':identity,'job':job,
                                     'reason':str(exc)[:500]}
                model_error('diffusion',job['sample_id'],exc)
            save_json(manifest_path,records)
    finally:
        del pipe
        release()
    return records

DIFFUSION_RESULTS = run_diffusion(DIFFUSION_JOBS,CFG) if CFG.RUN_DIFFUSION else {}
print(pd.Series([r['status'] for r in DIFFUSION_RESULTS.values()]).value_counts().to_dict())

In [ ]:
# 09 — Visualize source / original target / all generated candidates, with exact prompts

def preview_diffusion(sample_id, max_results=6):
    pair = next(p for p in PAIRS if p['sample_id']==sample_id)
    rows = [v for v in DIFFUSION_RESULTS.values()
            if v['status']=='ok' and v['job']['sample_id']==sample_id][:max_results]
    images = [('SOURCE',output_image(pair['source'])),('ORIGINAL TARGET',output_image(pair['target']))]
    images += [(f'{r["job"]["actual_words"]} words / strength={r["job"]["strength"]}',
                output_image(r['image'])) for r in rows]
    fig,axes = plt.subplots(1,len(images),figsize=(4*len(images),4),squeeze=False)
    for ax,(title,im) in zip(axes[0],images):
        ax.imshow(im);ax.set_title(title,fontsize=9);ax.axis('off')
    plt.tight_layout(); plt.show()
    print('FULL TARGET CAPTION:', CAPTIONS[sample_id]['target_caption'])
    for r in rows:
        j = r['job'];print(' ',j['actual_words'],'words; SD tokens:',r['sd_token_count'],
                         'truncated:',r['sd_truncated'],'|',j['prompt'])

if DIFFUSION_RESULTS:
    preview_diffusion(PAIRS[0]['sample_id'])

## C. Shared maps, region boxes, and numerical diagnostics

**Metrics:** normalized spatial entropy (0=concentrated, 1=uniform), mass in top 10% pixels, attention–gradient Spearman correlation, intersection-over-union of top-quantile connected-component boxes, and concept sensitivity via CLIP image-text cosine. Absolute map magnitudes from different models are **not directly comparable**; compare spatial distributions only after normalization and record the model-specific mechanism.

In [ ]:
# 10 — Map math / boxes / IoU (unit-testable pure functions)
from scipy.stats import spearmanr
import cv2

MAPS_DIR = ARTIFACTS/'maps'; MAPS_DIR.mkdir(exist_ok=True)
MAP_ROWS = []

def norm_map(arr):
    x = np.asarray(arr,dtype=np.float32)
    x = np.nan_to_num(x,nan=0,posinf=0,neginf=0)
    x = np.maximum(x,0)
    hi = float(x.max()) if x.size else 0.0
    return x/hi if hi>1e-12 else np.zeros_like(x)

def resize_map(arr,side=224):
    return norm_map(cv2.resize(np.asarray(arr,dtype=np.float32),(side,side),interpolation=cv2.INTER_LINEAR))

def normalized_entropy(arr):
    a = norm_map(arr).ravel().astype(np.float64)
    if not np.any(a): return float('nan')
    p = a/(a.sum()+1e-12)
    return float(-(p*np.log(p+1e-12)).sum()/np.log(len(p)))

def top_mass(arr,frac=.10):
    a=norm_map(arr).ravel()
    if a.sum()<=0: return float('nan')
    n=max(1,math.ceil(len(a)*frac))
    return float(np.sort(a)[-n:].sum()/a.sum())

def heatmap_boxes(heatmap,quantile=.80,max_boxes=3,min_area_frac=.005):
    a = norm_map(heatmap)
    if not a.any(): return []
    threshold = np.quantile(a[a>0], quantile)
    binary=(a>=threshold).astype(np.uint8)
    count, labels, stats, _ = cv2.connectedComponentsWithStats(binary,connectivity=8)
    min_area=max(2,round(min_area_frac*a.size))
    candidates=[]
    for k in range(1,count):
        x,y,w,h,area=map(int,stats[k])
        if area>=min_area:
            energy=float(a[labels==k].sum())
            candidates.append({'xyxy':[x,y,x+w,y+h], 'area':area, 'energy':energy})
    return sorted(candidates,key=lambda b:b['energy'],reverse=True)[:max_boxes]

def box_iou(a,b):
    xa,ya,Xa,Ya=a; xb,yb,Xb,Yb=b
    inter=max(0,min(Xa,Xb)-max(xa,xb))*max(0,min(Ya,Yb)-max(ya,yb))
    union=(Xa-xa)*(Ya-ya)+(Xb-xb)*(Yb-yb)-inter
    return float(inter/union) if union>0 else 0.0

def best_box_iou(a,b):
    return max([box_iou(x['xyxy'],y['xyxy']) for x in a for y in b], default=float('nan'))

def map_statistics(attn,grad):
    attn,grad=resize_map(attn),resize_map(grad)
    corr=float(spearmanr(attn.ravel(),grad.ravel()).statistic) if np.std(attn)>0 and np.std(grad)>0 else float('nan')
    boxes_a=heatmap_boxes(attn,CFG.HEATMAP_QUANTILE,CFG.MAX_BOXES)
    boxes_g=heatmap_boxes(grad,CFG.HEATMAP_QUANTILE,CFG.MAX_BOXES)
    return {'attn_entropy':normalized_entropy(attn),'grad_entropy':normalized_entropy(grad),
            'attn_top10_mass':top_mass(attn),'grad_top10_mass':top_mass(grad),
            'attn_grad_spearman':corr,'box_iou':best_box_iou(boxes_a,boxes_g),
            'boxes_attention':boxes_a, 'boxes_gradient':boxes_g}

def draw_map_overlay(img,heatmap,boxes=None,alpha=.55,side=224):
    image = square_preview(img,side)
    heat = resize_map(heatmap,side)
    color = (plt.get_cmap('magma')(heat)[...,:3]*255).astype(np.uint8)
    blended = Image.blend(image,Image.fromarray(color),alpha)
    pen = ImageDraw.Draw(blended)
    for item in boxes or []:
        pen.rectangle(item['xyxy'],outline=(0,255,180),width=2)
    return blended

def show_map_panel(img,attn,grad,title=''):
    metrics=map_statistics(attn,grad)
    items=[('Image',square_preview(img)),
           ('Attention + boxes',draw_map_overlay(img,attn,metrics['boxes_attention'])),
           ('Gradient + boxes',draw_map_overlay(img,grad,metrics['boxes_gradient']))]
    fig,axes=plt.subplots(1,3,figsize=(12,4))
    for ax,(name,view) in zip(axes,items):
        ax.imshow(view); ax.set_title(name);ax.axis('off')
    fig.suptitle(title);plt.tight_layout();plt.show()
    print({k:round(v,4) for k,v in metrics.items() if isinstance(v,float)})
    return metrics

def save_heatmaps(model_key, sid, role, concept, attention,gradient,metadata=None):
    slug = hashlib.sha1(f'{model_key}|{sid}|{role}|{concept}'.encode()).hexdigest()[:14]
    path=MAPS_DIR/f'{model_key}_{sid}_{slug}.npz'
    np.savez_compressed(path,attention=resize_map(attention),gradient=resize_map(gradient))
    row={'model':model_key,'sample_id':sid,'role':role,'concept':concept,'map_file':str(path),
         **map_statistics(attention,gradient), **(metadata or {})}
    MAP_ROWS.append(row)
    return row

## D. CLIP: image attention + caption/concept gradients + text-token attribution

Run ViT-B/16 and ViT-B/32 using Hugging Face `CLIPModel` with eager attention. Run **LAION ViT-g/14** using OpenCLIP (`ViT-g-14`, **not** `ViT-bigG-14`); the last visual block's attention (PyTorch MHA or OpenCLIP custom attention) matrices are reconstructed from its Q/K projections without modifying pretrained weights. Output image gradients are taken relative to the **normalized CLIP preprocessing input**, and caption/text-token sensitivity is from the gradient of cosine similarity. A separate leave-one-phrase-out (LOO) calculation estimates which concepts most influence the image–caption score.

**Critical:** CLIP vision attention is independent of caption tokens, so `concept` rows will share the same image-attention map for that image. Gradient maps will differ. Attention is measured at the **last vision layer's CLS query**, not attention rollout over all layers.

In [ ]:
# 11 — CLIP configuration, text ablation helpers
CLIP_SPECS = {
    'clip_b16': {'backend':'hf','id':'openai/clip-vit-base-patch16'},
    'clip_b32': {'backend':'hf','id':'openai/clip-vit-base-patch32'},
    'clip_g14': {'backend':'open_clip','arch':'ViT-g-14','pretrained':'laion2b_s12b_b42k'},
}

def phrase_removed(sentence,phrase):
    result = re.sub(r'(?<!\w)'+re.escape(phrase)+r'(?!\w)', '', sentence, count=1,flags=re.I)
    return re.sub(r'\s+',' ',re.sub(r'\s+([,.!?;:])',r'\1',result)).strip(' ,;') or 'photo'

def clip_features_similarity(image_features,text_features):
    return (F.normalize(image_features.float(),dim=-1)*
            F.normalize(text_features.float(),dim=-1)).sum(dim=-1)

def last_cls_patch_attention(tensor):
    # tensor is batch,heads,seq,seq; first token is ViT CLS
    matrix=tensor.float().detach()
    patch=matrix[:, :, 0, 1:].mean(dim=1)[0]
    count=int(patch.numel()); width=math.isqrt(count)
    if width*width!=count:
        raise ValueError(f'Non-square patch sequence: {count}')
    return patch.reshape(width,width).cpu().numpy()

def token_gradient_magnitude(grad,attention_mask):
    if grad is None: return []
    magn=grad.detach().float().norm(dim=-1)[0].cpu().numpy()
    used=attention_mask[0].detach().cpu().numpy().astype(bool)
    return magn[used].tolist()

In [ ]:
# 12 — HF CLIP: capture both visual/text self-attention and differentiable input
from transformers import CLIPModel, CLIPProcessor

class HFClipDiagnostic:
    def __init__(self,checkpoint,device=GPU0):
        self.device=device
        self.processor=CLIPProcessor.from_pretrained(checkpoint)
        self.model=CLIPModel.from_pretrained(checkpoint,attn_implementation='eager',
                                              torch_dtype=torch.float32).eval().to(device)
        self.model.requires_grad_(False)

    def tokenize(self, texts):
        return self.processor.tokenizer(texts,padding=True,truncation=True,
               max_length=77,return_tensors='pt').to(self.device)

    def encode_image(self, pixels):
        with torch.no_grad(): return F.normalize(self.model.get_image_features(pixel_values=pixels),dim=-1)

    def encode_text(self, texts):
        with torch.no_grad():
            batch=self.tokenize(texts)
            return F.normalize(self.model.get_text_features(**batch),dim=-1)

    def preprocess(self,image):
        return self.processor(images=image,return_tensors='pt')['pixel_values'].to(self.device)

    def explain(self,image,caption):
        pixels=self.preprocess(image).detach().requires_grad_(True)
        tokens=self.tokenize([caption])
        holder={}
        def capture(_,inputs,output):
            # Token embedding output becomes a differentiable leaf even though weights are frozen.
            value=output.detach().requires_grad_(True)
            value.retain_grad();holder['embed']=value
            return value
        hook=self.model.text_model.embeddings.token_embedding.register_forward_hook(capture)
        try:
            self.model.zero_grad(set_to_none=True)
            result=self.model(pixel_values=pixels,input_ids=tokens.input_ids,
                    attention_mask=tokens.attention_mask,output_attentions=True,return_dict=True)
            score=clip_features_similarity(result.image_embeds,result.text_embeds).sum()
            attn=last_cls_patch_attention(result.vision_model_output.attentions[-1])
            txt_attn=result.text_model_output.attentions[-1].detach().float()[0].mean(dim=0)
            text_query=int(tokens.attention_mask[0].sum().item())-1
            scores=txt_attn[text_query].cpu().numpy()
            score.backward()
            grad=pixels.grad.detach().float().abs().mean(dim=1)[0].cpu().numpy()
            token_grad=token_gradient_magnitude(holder['embed'].grad,tokens.attention_mask)
            ids=tokens.input_ids[0][tokens.attention_mask[0].bool()].tolist()
            strings=self.processor.tokenizer.convert_ids_to_tokens(ids)
            return {'attention':attn,'gradient':grad,'similarity':float(score.detach()),
                    'tokens':strings,'token_grad':token_grad,'text_attention':scores.tolist()}
        finally:
            hook.remove()
            self.model.zero_grad(set_to_none=True)

    def sensitivity(self,image,caption,concepts):
        with torch.no_grad():
            pixel=self.preprocess(image)
            image_f=self.encode_image(pixel)
            captions=[caption]+[phrase_removed(caption,c) for c in concepts]
            all_text=self.encode_text(captions)
            similarities=(image_f @ all_text.T)[0].float().cpu().numpy()
        return [{'concept':c,'caption_similarity':float(similarities[0]),
                 'loo_delta':float(similarities[0]-similarities[i+1])}
                for i,c in enumerate(concepts)]

    def close(self):
        del self.model,self.processor
        release()

In [ ]:
# 13 — OpenCLIP LAION ViT-g/14: native/custom visual attention + gradients
import open_clip

class OpenClipDiagnostic:
    def __init__(self,arch,pretrained,device=GPU1):
        self.device=device
        self.model,_,self.preprocess_fn=open_clip.create_model_and_transforms(arch,pretrained=pretrained)
        self.model=self.model.to(device=device,dtype=torch.float16).eval()
        self.model.requires_grad_(False)
        self.tokenizer=open_clip.get_tokenizer(arch)
        self.mha=self.model.visual.transformer.resblocks[-1].attn
        if not hasattr(self.mha,'in_proj_weight') or not hasattr(self.mha,'num_heads'):
            raise TypeError(f'Unsupported OpenCLIP attention module: {type(self.mha)}')

    def preprocess(self,image):
        return self.preprocess_fn(image).unsqueeze(0).to(self.device,dtype=torch.float16)

    def encode_image(self,pixels):
        return F.normalize(self.model.encode_image(pixels).float(),dim=-1)

    def encode_text(self,texts):
        tokens=self.tokenizer(texts).to(self.device)
        return F.normalize(self.model.encode_text(tokens).float(),dim=-1)

    @staticmethod
    def compute_mha_attention(mha,query,key):
        # Compatible with legacy torch MultiheadAttention AND OpenCLIP v3 Attention.
        # OpenCLIP custom Attention uses batch-first and optional q/k norm.
        is_native=isinstance(mha,torch.nn.MultiheadAttention)
        if is_native and not mha.batch_first:
            query,key=query.transpose(0,1),key.transpose(0,1)
        if mha.in_proj_weight is None:
            raise NotImplementedError('Cross-dimensional q/k projection not supported')
        embed=mha.in_proj_weight.shape[0]//3
        heads=mha.num_heads;dim=embed//heads
        q=F.linear(query,mha.in_proj_weight[:embed],
                   mha.in_proj_bias[:embed] if mha.in_proj_bias is not None else None)
        k=F.linear(key,mha.in_proj_weight[embed:2*embed],
                   mha.in_proj_bias[embed:2*embed] if mha.in_proj_bias is not None else None)
        q=q.reshape(q.shape[0],q.shape[1],heads,dim).permute(0,2,1,3)
        k=k.reshape(k.shape[0],k.shape[1],heads,dim).permute(0,2,1,3)
        if not is_native:
            q=mha.ln_q(q)
            k=mha.ln_k(k)
        if not is_native and getattr(mha,'logit_scale',None) is not None:
            logits=(F.normalize(q.float(),dim=-1) @
                    F.normalize(k.float(),dim=-1).transpose(-1,-2))
            scale=mha.logit_scale.float().clamp(max=mha.logit_scale_max).exp()
            logits=logits*scale
        else:
            logits=(q.float() @ k.float().transpose(-1,-2))/math.sqrt(dim)
        return torch.softmax(logits,dim=-1)

    def explain(self,image,caption):
        pixels=self.preprocess(image).detach().requires_grad_(True)
        toks=self.tokenizer([caption]).to(self.device)
        recorded={}
        def attention_input(mod,args):
            recorded['vision_input'] = args[0].detach()
        def text_embeddings(mod,inputs,output):
            leaf=output.detach().requires_grad_(True)
            leaf.retain_grad();recorded['text_embed']=leaf
            return leaf
        hooks=[self.mha.register_forward_pre_hook(attention_input),
               self.model.token_embedding.register_forward_hook(text_embeddings)]
        try:
            self.model.zero_grad(set_to_none=True)
            image_f=self.encode_image(pixels)
            text_f=F.normalize(self.model.encode_text(toks).float(),dim=-1)
            score=(image_f*text_f).sum()
            activation=recorded['vision_input']
            attn=self.compute_mha_attention(self.mha,activation,activation).detach()
            vision=last_cls_patch_attention(attn)
            (score*1024.0).backward()
            pixel_grad=(pixels.grad.detach().float()/1024.0).abs().mean(dim=1)[0].cpu().numpy()
            grad=recorded['text_embed'].grad
            ids=toks[0].tolist()
            valid=sum(int(t!=0) for t in ids)
            strings=[self.tokenizer.decode([z]) for z in ids[:valid]]
            norms=(grad.detach().float()/1024.0).norm(dim=-1)[0][:valid].cpu().tolist()
            return {'attention':vision,'gradient':pixel_grad,'similarity':float(score.detach()),
                    'tokens':strings,'token_grad':norms,'text_attention':[]}
        finally:
            for hook in hooks:hook.remove()
            recorded.clear()
            self.model.zero_grad(set_to_none=True)

    def sensitivity(self,image,caption,concepts):
        with torch.no_grad():
            image_f=self.encode_image(self.preprocess(image))
            texts=[caption]+[phrase_removed(caption,c) for c in concepts]
            scores=(image_f@self.encode_text(texts).T)[0].float().cpu().numpy()
        return [{'concept':c,'caption_similarity':float(scores[0]),
                 'loo_delta':float(scores[0]-scores[i+1])} for i,c in enumerate(concepts)]

    def close(self):
        del self.model,self.preprocess_fn,self.mha,self.tokenizer
        release()

In [ ]:
# 14 — Select image roles and persisted analysis records
# Analyse source and target on all 10 pairs. Add ONE matching diffusion result per sample,
# to avoid multiplying gradients by all caption-length sweep candidates.

def images_for_analysis(pair,diffusion_results):
    views={'source':output_image(pair['source']),'target':output_image(pair['target'])}
    candidates=[r for r in diffusion_results.values()
                if r['status']=='ok' and r['job']['sample_id']==pair['sample_id']]
    if candidates:
        # Prefer the baseline 0.7 / 24 words / first seed, otherwise nearest candidate.
        selected=min(candidates,key=lambda r:(abs(r['job']['strength']-0.7),
                                              abs(r['job']['actual_words']-24),r['job']['seed']))
        views['generated']=output_image(selected['image'])
    return views

CLIP_ROWS=[]; TOKEN_ROWS=[]; LOO_ROWS=[]

def append_clip_outputs(model_key, sample_id, role, prompt_name,expl):
    row=save_heatmaps(model_key,sample_id,role,prompt_name,
                      expl['attention'],expl['gradient'],{'cosine':expl['similarity'],
                                                         'mechanism':'CLIP vision CLS self-attention / pixel grad'})
    for i,token in enumerate(expl['tokens']):
        TOKEN_ROWS.append({'model':model_key,'sample_id':sample_id,'role':role,
                           'prompt':prompt_name,'position':i,'token':str(token),
                           'grad_norm':expl['token_grad'][i] if i<len(expl['token_grad']) else np.nan,
                           'text_self_attention':expl['text_attention'][i] if i<len(expl['text_attention']) else np.nan})
    CLIP_ROWS.append(row)

def build_clip_diagnostic(model_key):
    spec=CLIP_SPECS[model_key]
    if spec['backend']=='hf': return HFClipDiagnostic(spec['id'],GPU0)
    return OpenClipDiagnostic(spec['arch'],spec['pretrained'],GPU1)

def run_clip_analysis(pairs,captions,concept_records,cfg):
    for model_key in cfg.CLIP_MODELS:
        print('\n### MODEL:',model_key)
        wrapper=None
        try:
            wrapper=build_clip_diagnostic(model_key)
            for pair in tqdm(pairs,desc=model_key):
                sid=pair['sample_id']; caption=captions[sid]['target_caption']
                concepts=concept_records[sid]['concepts'][:cfg.CLIP_MAX_CONCEPTS]
                for role,img in images_for_analysis(pair,DIFFUSION_RESULTS).items():
                    # Explicitly save caption map then concept-specific pixel-grad maps.
                    try:
                        prompts=[('caption',caption)]+[(f'concept:{c}',c) for c in concepts]
                        for label,prompt in prompts:
                            expl=wrapper.explain(img,prompt)
                            append_clip_outputs(model_key,sid,role,label,expl)
                        for r in wrapper.sensitivity(img,caption,concepts):
                            LOO_ROWS.append({'model':model_key,'sample_id':sid,'role':role,**r})
                    except Exception as exc: model_error(model_key,sid+'_'+role,exc)
                pd.DataFrame(LOO_ROWS).to_csv(ARTIFACTS/'concept_loo.csv',index=False)
                pd.DataFrame(TOKEN_ROWS).to_csv(ARTIFACTS/'token_gradients.csv',index=False)
                save_json(ARTIFACTS/'clip_map_records.json',CLIP_ROWS)
            RUN_STATUS.append({'model':model_key,'sample_id':'all','status':'ok'})
        except Exception as exc: model_error(model_key,'model_load',exc)
        finally:
            if wrapper is not None:wrapper.close()
            release()

if CFG.RUN_CLIP:
    run_clip_analysis(PAIRS,CAPTIONS,CONCEPT_RECORDS,CFG)
print('Maps:',len(MAP_ROWS),'token rows:',len(TOKEN_ROWS),'LOO rows:',len(LOO_ROWS))

## E. BLIP-2: Q-Former cross-attention and caption-conditioned loss gradients

The BLIP-2 OPT-2.7B backbone is memory intensive. Use **4-bit** quantization for the frozen LM, allow placement across 2 GPUs, and process **one image at a time**. Query→vision cross-attention is extracted from BLIP-2's Q-Former via `output_attentions=True`. For each concept the optional gradient is computed from the **image-conditioned log-likelihood of that phrase**, through the BLIP-2 forward `labels` argument. Note that Q-Former attention itself stays unchanged across concepts. Explicit error status is saved if the quantized backward path is unsupported.

In [ ]:
# 15 — BLIP-2 loader and Q-Former diagnostic
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig

class Blip2Diagnostic:
    def __init__(self,checkpoint):
        self.processor=Blip2Processor.from_pretrained(checkpoint)
        qconf=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_compute_dtype=torch.float16,
                                bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True)
        self.model=Blip2ForConditionalGeneration.from_pretrained(checkpoint,
              quantization_config=qconf,device_map='auto',low_cpu_mem_usage=True,
              torch_dtype=torch.float16).eval()
        self.model.requires_grad_(False)

    def prepare(self,image):
        batch=self.processor(images=image,return_tensors='pt')
        batch['pixel_values']=batch['pixel_values'].to(first_device(self.model),dtype=torch.float16)
        return batch

    def qformer_attention(self,pixels):
        model=self.model
        with torch.inference_mode():
            vision=model.vision_model(pixel_values=pixels,return_dict=True)
            features=vision.last_hidden_state
            mask=torch.ones(features.size()[:-1],device=features.device,dtype=torch.long)
            query=model.query_tokens.expand(features.shape[0],-1,-1)
            qdevice=first_device(model.qformer)
            outputs=model.qformer(query_embeds=query.to(qdevice),
                     encoder_hidden_states=features.to(qdevice),encoder_attention_mask=mask.to(qdevice),
                     output_attentions=True,return_dict=True)
            layers=[z for z in outputs.cross_attentions if z is not None]
            if not layers: raise RuntimeError('Q-Former returned no cross-attention tensors')
            cross=layers[-1].detach().float()[0].mean(dim=(0,1))
            # BLIP-2 ViT image patches normally [CLS, 16x16 patch tokens, ...].
            length=cross.shape[0];n=math.isqrt(length-1)
            if n*n==length-1: cross=cross[1:]
            else:
                n=math.isqrt(length)
                if n*n!=length: raise RuntimeError(f'Unsupported BLIP2 visual token layout: {length}')
            return cross.reshape(n,n).cpu().numpy()

    def caption_loss_gradient(self,image,caption):
        batch=self.prepare(image)
        pixels=batch['pixel_values'].detach().requires_grad_(True)
        # Text is teacher-forced in an OPT causal language model. Positive image
        # sensitivity = gradient magnitude of the negative log likelihood of caption.
        text=self.processor.tokenizer(caption,return_tensors='pt',add_special_tokens=True)
        text={k:v.to(first_device(self.model.language_model)) for k,v in text.items()}
        self.model.zero_grad(set_to_none=True)
        outputs=self.model(pixel_values=pixels, input_ids=text['input_ids'],
                           attention_mask=text['attention_mask'], labels=text['input_ids'])
        if outputs.loss is None: raise RuntimeError('BLIP2 did not return a caption loss')
        outputs.loss.backward()
        if pixels.grad is None: raise RuntimeError('No BLIP2 pixel gradient')
        grad=pixels.grad.detach().float().abs().mean(dim=1)[0].cpu().numpy()
        return grad,float(outputs.loss.detach())

    def explain(self,image,caption,compute_grad):
        batch=self.prepare(image)
        attn=self.qformer_attention(batch['pixel_values'])
        grad=np.zeros_like(attn);loss=np.nan;grad_error=None
        if compute_grad:
            try:
                grad,loss=self.caption_loss_gradient(image,caption)
            except Exception as exc:
                grad_error=f'{type(exc).__name__}: {exc}'
                print('[BLIP2] Caption gradient unavailable:',grad_error[:220])
        return {'attention':attn,'gradient':grad,'loss':loss,
                'gradient_valid': bool(compute_grad and grad_error is None),
                'gradient_error':grad_error}

    def close(self):
        del self.model,self.processor
        release()

In [ ]:
# 16 — BLIP-2 sequential 10-pair execution (skip grad separately if it fails)
BLIP2_ROWS=[]

def run_blip2(pairs,captions,concept_records,cfg):
    adapter=None
    try:
        adapter=Blip2Diagnostic(cfg.BLIP2_ID)
        for pair in tqdm(pairs,desc='BLIP2 Q-Former'):
            sid=pair['sample_id']
            concepts=concept_records[sid]['concepts'][:cfg.CLIP_MAX_CONCEPTS]
            # BLIP-2 only target and generated; source can be reenabled if needed.
            for role,img in images_for_analysis(pair,DIFFUSION_RESULTS).items():
                if role=='source': continue
                for name,text in [('caption',captions[sid]['target_caption'])]+[(f'concept:{c}',c) for c in concepts]:
                    try:
                        expl=adapter.explain(img,text,cfg.BLIP2_ENABLE_GRAD)
                        row=save_heatmaps('blip2',sid,role,name,expl['attention'],expl['gradient'],
                             {'caption_nll':expl['loss'],'gradient_valid':expl['gradient_valid'],
                              'gradient_error':expl['gradient_error'],
                              'mechanism':'BLIP2 Q-Former cross-attention / caption NLL pixel grad'})
                        BLIP2_ROWS.append(row)
                    except Exception as exc: model_error('blip2',sid+'_'+role+'_'+name,exc)
            save_json(ARTIFACTS/'blip2_map_records.json',BLIP2_ROWS)
        RUN_STATUS.append({'model':'blip2','sample_id':'all','status':'complete',
                           'note':'Check per-row gradient_valid before treating gradient maps as valid'})
    except Exception as exc:model_error('blip2','model_load',exc)
    finally:
        if adapter is not None:adapter.close()
        release()

if CFG.RUN_BLIP2: run_blip2(PAIRS,CAPTIONS,CONCEPT_RECORDS,CFG)

## F. Qwen3-VL: concept-query → image-token attention and gradient

The **8B** instruct model is quantized in NF4 and dispatched across both GPUs. The question is `Is <concept> visible in this image? Answer yes or no.` The diagnostic scalar is the difference between the **first-token** logits for `yes` and `no` (not a calibrated object probability). The last query-token attention to image tokens is taken from the **final self-attention layer**, averaged over heads. A separate optional derivative computes `d(logit_yes - logit_no)/d(pixel_values)`; Qwen image `pixel_values` are **patchified vision inputs**, not RGB pixels. To compare with CLIP the patchified gradient is unshuffled and resized. If the attention/token geometry does not match, fail explicitly rather than invent a heatmap.

Running Qwen3-VL for ten pairs, two concepts each, may be slow on dual T4 GPUs. This cell is independent of diffusion; it does not need to hold any CLIP/BLIP2 model weights in memory.

In [ ]:
# 17 — Qwen3-VL model loading / pixel and token alignment
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

class Qwen3VLDiagnostic:
    def __init__(self,checkpoint,cfg):
        self.processor=AutoProcessor.from_pretrained(checkpoint,
                   min_pixels=64*28*28,max_pixels=cfg.QWEN_MAX_PIXELS)
        qconf=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
                bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
        # eager attention is required; SDPA/flash kernels do not expose matrices.
        self.model=Qwen3VLForConditionalGeneration.from_pretrained(
            checkpoint,quantization_config=qconf,device_map='auto',
            attn_implementation='eager',low_cpu_mem_usage=True,
            torch_dtype=torch.float16).eval()
        self.model.requires_grad_(False)

    def inputs(self,image,concept):
        question=f'Is {concept} visible in this image? Answer yes or no.'
        content=[{'type':'image','image':image},{'type':'text','text':question}]
        messages=[{'role':'user','content':content}]
        data=self.processor.apply_chat_template(messages,tokenize=True,
                 add_generation_prompt=True,return_dict=True,return_tensors='pt')
        entry=first_device(self.model)
        return {k:(v.to(entry) if isinstance(v,torch.Tensor) else v) for k,v in data.items()}

    def image_positions(self,inputs):
        ids=inputs['input_ids'][0]
        image_id=getattr(self.model.config,'image_token_id',None)
        if image_id is None:image_id=getattr(self.processor,'image_token_id',None)
        if image_id is not None:
            ix=(ids==int(image_id)).nonzero(as_tuple=False).flatten()
        elif 'mm_token_type_ids' in inputs:
            ix=(inputs['mm_token_type_ids'][0]==1).nonzero(as_tuple=False).flatten()
        else: raise RuntimeError('Cannot locate Qwen image tokens')
        if not len(ix):raise RuntimeError('Found zero image tokens')
        return ix

    def grid(self,inputs):
        if 'image_grid_thw' not in inputs: raise RuntimeError('No image_grid_thw metadata')
        t,h,w=(int(z) for z in inputs['image_grid_thw'][0].tolist())
        m=int(getattr(self.model.config.vision_config,'spatial_merge_size',2))
        if h%m or w%m:raise RuntimeError(f'Grid {h}x{w} not divisible by merge {m}')
        return t,h,w,m

    def token_to_map(self,scores,inputs):
        t,h,w,m=self.grid(inputs)
        gh,gw=h//m,w//m
        if scores.numel()!=t*gh*gw:
            raise RuntimeError(f'Qwen text→image count {scores.numel()} != grid {t*gh*gw}')
        return scores.reshape(t,gh,gw).float().mean(dim=0).cpu().numpy()

    def patch_to_map(self,patch_scores,inputs):
        t,h,w,m=self.grid(inputs)
        if patch_scores.numel()!=t*h*w:
            raise RuntimeError(f'Qwen vision pixel patches {patch_scores.numel()} != {t*h*w}')
        # Qwen spatial-merge packing uses [T,H/m,W/m,m,m,...]; unshuffle.
        reconstructed=patch_scores.reshape(t,h//m,w//m,m,m).permute(0,1,3,2,4).reshape(t,h,w)
        # Match merged language-model visual token grid for attention correlation.
        return reconstructed.mean(dim=0).reshape(h//m,m,w//m,m).mean(dim=(1,3)).float().cpu().numpy()

    def answer_ids(self):
        tok=self.processor.tokenizer
        yes=tok.encode(' yes',add_special_tokens=False)
        no=tok.encode(' no',add_special_tokens=False)
        if not yes or not no:raise ValueError('yes/no tokenizer produced empty IDs')
        return yes[0],no[0]

    def explain(self,image,concept,with_grad=True):
        inputs=self.inputs(image,concept)
        image_ix=self.image_positions(inputs)
        pixels=inputs['pixel_values'].detach().requires_grad_(bool(with_grad))
        inputs['pixel_values']=pixels
        yes,no=self.answer_ids()
        self.model.zero_grad(set_to_none=True)
        # Full eager attentions at small image resolution: last-token query can see image tokens.
        with torch.enable_grad() if with_grad else torch.inference_mode():
            out=self.model(**inputs,output_attentions=True,return_dict=True,
                           use_cache=False,logits_to_keep=1)
            raw=out.logits[0,-1]
            scalar=raw[yes].float()-raw[no].float()
            layers=out.attentions
            if not layers or layers[-1] is None:
                raise RuntimeError('No Qwen3-VL attention: ensure eager implementation')
            full_attention=layers[-1][0].detach().float().mean(dim=0)
            image_ix=image_ix.to(full_attention.device)
            last=full_attention[-1,image_ix]
            attn=self.token_to_map(last,inputs)
            token_diagnostics=[]
            after_image=int(image_ix.max().item())+1
            token_ids=inputs['input_ids'][0]
            for position in range(after_image,token_ids.numel()):
                # Per-text-token mass to image tokens. It is NOT per-token pixel gradient.
                vector=full_attention[position,image_ix]
                token_map=self.token_to_map(vector,inputs)
                text=self.processor.tokenizer.convert_ids_to_tokens(int(token_ids[position]))
                token_diagnostics.append({
                    'position':position,'token':text,
                    'image_attention_mass':float(vector.sum()),
                    'image_attention_entropy':normalized_entropy(token_map),
                    'image_attention_top10_mass':top_mass(token_map),
                    'spatial_map':token_map})
            grad=np.zeros_like(attn)
            if with_grad:
                scalar.backward()
                if pixels.grad is None:raise RuntimeError('Qwen vision patch grad missing')
                grad=self.patch_to_map(pixels.grad.detach().float().abs().mean(dim=-1),inputs)
            margin=float(scalar.detach())
        self.model.zero_grad(set_to_none=True)
        return {'attention':attn,'gradient':grad,'yes_no_margin':margin,
                'image_tokens':len(image_ix),'grid':self.grid(inputs),
                'token_diagnostics':token_diagnostics}

    def close(self):
        del self.model,self.processor
        release()

In [ ]:
# 18 — Run quantized Qwen3-VL on the first two caption-extracted concepts per image
QWEN_ROWS=[]; QWEN_TOKEN_ROWS=[]

def run_qwen(pairs,concept_records,cfg):
    adapter=None
    try:
        adapter=Qwen3VLDiagnostic(cfg.QWEN_ID,cfg)
        for pair in tqdm(pairs,desc='Qwen3-VL'):
            sid=pair['sample_id']; concepts=concept_records[sid]['concepts'][:cfg.QWEN_MAX_CONCEPTS]
            for role,img in images_for_analysis(pair,DIFFUSION_RESULTS).items():
                if role=='source': continue
                for concept in concepts:
                    try:
                        gradient_error=None
                        try:
                            result=adapter.explain(img,concept,cfg.QWEN_ENABLE_GRAD)
                            gradient_valid=bool(cfg.QWEN_ENABLE_GRAD)
                        except Exception as exc:
                            if not cfg.QWEN_ENABLE_GRAD:raise
                            gradient_error=f'{type(exc).__name__}: {exc}'
                            print('[Qwen3VL] Gradient failed; retrying attention-only:', gradient_error[:180])
                            release()
                            result=adapter.explain(img,concept,False)
                            gradient_valid=False
                        record=save_heatmaps('qwen3vl',sid,role,f'concept:{concept}',
                                            result['attention'],result['gradient'],
                            {'yes_no_margin':result['yes_no_margin'],
                             'image_tokens':result['image_tokens'],
                             'grid':str(result['grid']),
                             'gradient_valid':gradient_valid,'gradient_error':gradient_error,
                             'mechanism':'Qwen last-text-token image attention / patchified image logit grad'})
                        QWEN_ROWS.append(record)
                        key=hashlib.sha1(f'{sid}/{role}/{concept}'.encode()).hexdigest()[:12]
                        token_file=MAPS_DIR/f'qwen3vl_tokens_{key}.npz'
                        all_maps=np.stack([resize_map(t['spatial_map'])
                                           for t in result['token_diagnostics']])
                        np.savez_compressed(token_file,heatmaps=all_maps)
                        for index,token_record in enumerate(result['token_diagnostics']):
                            QWEN_TOKEN_ROWS.append({'sample_id':sid,'role':role,'concept':concept,
                              'position':token_record['position'],'token':token_record['token'],
                              'image_attention_mass':token_record['image_attention_mass'],
                              'image_attention_entropy':token_record['image_attention_entropy'],
                              'image_attention_top10_mass':token_record['image_attention_top10_mass'],
                              'maps_npz':str(token_file),'map_index':index})
                    except Exception as exc: model_error('qwen3vl',sid+'_'+role+'_'+concept,exc)
            save_json(ARTIFACTS/'qwen_map_records.json',QWEN_ROWS)
            if QWEN_TOKEN_ROWS:
                pd.DataFrame(QWEN_TOKEN_ROWS).to_csv(ARTIFACTS/'qwen_text_token_attention.csv',index=False)
        RUN_STATUS.append({'model':'qwen3vl','sample_id':'all','status':'complete',
                           'note':'Check per-row gradient_valid before treating gradient maps as valid'})
    except Exception as exc: model_error('qwen3vl','model_load',exc)
    finally:
        if adapter is not None:adapter.close()
        release()

if CFG.RUN_QWEN:run_qwen(PAIRS,CONCEPT_RECORDS,CFG)

In [ ]:
# 18a — Identify important Qwen TEXT tokens by attention mass to image tokens
# (The word ranking is a diagnostic, NOT a learned concept binding or causal importance.)
def inspect_qwen_text_tokens(sample_id=None,role='target',concept=None,top=12):
    if not QWEN_TOKEN_ROWS:
        print('Qwen token attention unavailable; inspect run_status and model_status.csv')
        return
    sid=sample_id or PAIRS[0]['sample_id']
    concept=concept or CONCEPT_RECORDS[sid]['concepts'][0]
    frame=pd.DataFrame(QWEN_TOKEN_ROWS)
    frame=frame[(frame.sample_id==sid)&(frame.role==role)&(frame.concept==concept)]
    if frame.empty:
        print('No Qwen token maps for',sid,role,concept)
        return
    ranked=frame.sort_values('image_attention_mass',ascending=False).head(top)
    display(ranked[['position','token','image_attention_mass',
                    'image_attention_entropy','image_attention_top10_mass']])
    pair=next(p for p in PAIRS if p['sample_id']==sid)
    img=images_for_analysis(pair,DIFFUSION_RESULTS)[role]
    fig,axes=plt.subplots(1,min(top,len(ranked)),figsize=(3.5*min(top,len(ranked)),3.5),squeeze=False)
    for ax,(_,row) in zip(axes[0],ranked.iterrows()):
        with np.load(row['maps_npz']) as data:
            heat=data['heatmaps'][int(row['map_index'])]
        ax.imshow(draw_map_overlay(img,heat))
        ax.set_title(f'{row["token"]} | mass={row["image_attention_mass"]:.3f}')
        ax.axis('off')
    plt.tight_layout();plt.show()

if QWEN_TOKEN_ROWS:
    inspect_qwen_text_tokens(top=6)

## G. Inspection, paired plots, exports, reproducibility checks

Generated candidates are **not** conditioned on the target image pixels unless `DIFFUSION_INIT_IMAGE='target'`; the baseline uses the **source**. All maps are saved as compressed `.npz`, with `CSV` summaries and a machine-readable `run_status.json`. Inspect both “target” and “generated” to test whether the diffusion candidate actually shifts visual features toward target concepts. Differences between encoders should be interpreted in light of their **different** attention mechanisms.

In [ ]:
# 19 — Inspect a single model × sample × concept, directly from saved files

def select_map(model,sid,role='target',concept='caption'):
    hits=[r for r in MAP_ROWS if r['model']==model and r['sample_id']==sid and
          r['role']==role and r['concept']==concept]
    if not hits:raise KeyError(f'No map: {model} / {sid} / {role} / {concept}')
    return hits[-1]

def inspect_one(model='clip_b16',sample_id=None,role='target',concept='caption'):
    sample_id=sample_id or PAIRS[0]['sample_id']
    pair=next(p for p in PAIRS if p['sample_id']==sample_id)
    record=select_map(model,sample_id,role,concept)
    with np.load(record['map_file']) as tensors:
        att=tensors['attention'];grad=tensors['gradient']
    img=images_for_analysis(pair,DIFFUSION_RESULTS)[role]
    return show_map_panel(img,att,grad,f'{model} / {sample_id} / {role} / {concept}')

if MAP_ROWS:
    inspect_one(model=MAP_ROWS[0]['model'],sample_id=MAP_ROWS[0]['sample_id'],
                role=MAP_ROWS[0]['role'],concept=MAP_ROWS[0]['concept'])

In [ ]:
# 20 — Side-by-side: source vs target vs generated attention and gradient

def compare_roles(model,sid,concept='caption'):
    pair=next(p for p in PAIRS if p['sample_id']==sid)
    roles=images_for_analysis(pair,DIFFUSION_RESULTS)
    grid=[]
    for role,img in roles.items():
        try:
            record=select_map(model,sid,role,concept)
        except KeyError:continue
        with np.load(record['map_file']) as x:
            at,gr=x['attention'],x['gradient']
        stats=map_statistics(at,gr)
        grid.append((role,img,at,gr,stats))
    if not grid:
        print('No available maps for',model,sid,concept);return
    fig,axes=plt.subplots(len(grid),3,figsize=(12,3.6*len(grid)),squeeze=False)
    for i,(role,img,att,grad,stats) in enumerate(grid):
        views=[square_preview(img),
               draw_map_overlay(img,att,stats['boxes_attention']),
               draw_map_overlay(img,grad,stats['boxes_gradient'])]
        for k,(v,title) in enumerate(zip(views,['image','attention + box','gradient + box'])):
            axes[i,k].imshow(v)
            axes[i,k].set_title(f'{role} / {title}' if k==0 else title)
            axes[i,k].axis('off')
    fig.suptitle(f'{model} | {sid} | {concept}'); plt.tight_layout();plt.show()

if MAP_ROWS:
    compare_roles(MAP_ROWS[0]['model'], PAIRS[0]['sample_id'],'caption')

In [ ]:
# 21 — Compare models on identical image / same concept, annotate mechanism explicitly

def compare_models(sample_id,role='target',concept='caption'):
    pair=next(p for p in PAIRS if p['sample_id']==sample_id)
    image=images_for_analysis(pair,DIFFUSION_RESULTS)[role]
    models=list(dict.fromkeys(r['model'] for r in MAP_ROWS
                              if r['sample_id']==sample_id and r['role']==role and r['concept']==concept))
    if not models:
        print('No model results for',sample_id,role,concept);return
    fig,axes=plt.subplots(len(models),2,figsize=(9,3.3*len(models)),squeeze=False)
    for i,model in enumerate(models):
        row=select_map(model,sample_id,role,concept)
        with np.load(row['map_file']) as t:at,gr=t['attention'],t['gradient']
        m=map_statistics(at,gr)
        axes[i,0].imshow(draw_map_overlay(image,at,m['boxes_attention']))
        axes[i,1].imshow(draw_map_overlay(image,gr,m['boxes_gradient']))
        axes[i,0].set_title(f'{model}: attention');axes[i,1].set_title(f'{model}: gradient')
        for a in axes[i]:a.axis('off')
    plt.tight_layout();plt.show()

if MAP_ROWS:
    compare_models(PAIRS[0]['sample_id'],'target','caption')

In [ ]:
# 22a — Quantify caption budgets, source preservation and target likeness (CLIP B/16)
# This cell reloads a frozen CLIP B/16 diagnostic only if there are generated images.
DIFFUSION_SCORE_ROWS=[]

def diffusion_sweep_scores(pairs,captions,results):
    adapter=HFClipDiagnostic(CLIP_SPECS['clip_b16']['id'],GPU0)
    rows=[]
    try:
        for pair in tqdm(pairs,desc='Diffusion CLIP score sweep'):
            sid=pair['sample_id']; src=output_image(pair['source']);tgt=output_image(pair['target'])
            with torch.no_grad():
                emb_src=adapter.encode_image(adapter.preprocess(src))
                emb_tgt=adapter.encode_image(adapter.preprocess(tgt))
            for record in results.values():
                if record['status']!='ok' or record['job']['sample_id']!=sid:continue
                with torch.no_grad():
                    emb_img=adapter.encode_image(adapter.preprocess(output_image(record['image'])))
                    emb_txt=adapter.encode_text([record['job']['prompt']])
                    source_sim=float((emb_img*emb_src).sum())
                    target_sim=float((emb_img*emb_tgt).sum())
                    text_sim=float((emb_img*emb_txt).sum())
                rows.append({'sample_id':sid,'budget':record['job']['budget'],
                    'actual_words':record['job']['actual_words'],
                    'sd_tokens':record['sd_token_count'],
                    'sd_truncated':record['sd_truncated'],
                    'strength':record['job']['strength'], 'seed':record['job']['seed'],
                    'generated_source_cosine':source_sim,
                    'generated_target_cosine':target_sim,
                    'generated_prompt_cosine':text_sim,
                    'generated_path':record['image']})
        return rows
    finally:adapter.close()

if DIFFUSION_RESULTS and any(r['status']=='ok' for r in DIFFUSION_RESULTS.values()):
    DIFFUSION_SCORE_ROWS=diffusion_sweep_scores(PAIRS,CAPTIONS,DIFFUSION_RESULTS)
    table=pd.DataFrame(DIFFUSION_SCORE_ROWS)
    table.to_csv(CFG.OUTPUT_DIR/'diffusion_caption_sweep.csv',index=False)
    display(table.groupby(['actual_words','strength']).agg(
        n=('sample_id','count'), target_cosine=('generated_target_cosine','mean'),
        prompt_cosine=('generated_prompt_cosine','mean'),
        source_cosine=('generated_source_cosine','mean')).round(4))
    fig,ax=plt.subplots(figsize=(7,4))
    for strength,part in table.groupby('strength'):
        means=part.groupby('actual_words')['generated_target_cosine'].mean()
        ax.plot(means.index,means.values,'o-',label=f'strength={strength}')
    ax.set(xlabel='Actual caption word count',ylabel='CLIP B/16 cos(generated, target)',
           title='Descriptive metric only: not a diffusion quality guarantee')
    ax.legend();plt.tight_layout();plt.show()

In [ ]:
# 21a — Distribution analysis over the 10 samples: target vs generated concept maps
# Match (sample_id, model, exact concept). Do not compare absolute gradient amplitudes
# across independently normalized models; use relative concentration/entropy.

def compare_target_generated_maps(rows):
    frame=pd.DataFrame(rows)
    if frame.empty:return pd.DataFrame()
    out=[]
    groups=frame.groupby(['model','sample_id','concept'])
    for (model,sid,concept),subset in groups:
        targets=subset[subset.role=='target']
        generated=subset[subset.role=='generated']
        if len(targets)!=1 or len(generated)!=1:continue
        target,synthetic=targets.iloc[0],generated.iloc[0]
        if not bool(target.get('gradient_valid',True)) or not bool(synthetic.get('gradient_valid',True)):
            continue
        with np.load(target.map_file) as t, np.load(synthetic.map_file) as g:
            att_t,grad_t=t['attention'],t['gradient']
            att_g,grad_g=g['attention'],g['gradient']
        def correlation(a,b):
            if not np.std(a) or not np.std(b):return float('nan')
            return float(spearmanr(a.ravel(),b.ravel()).statistic)
        out.append({'model':model,'sample_id':sid,'concept':concept,
           'delta_grad_entropy':normalized_entropy(grad_g)-normalized_entropy(grad_t),
           'delta_grad_top10_mass':top_mass(grad_g)-top_mass(grad_t),
           'target_generated_attn_corr':correlation(att_t,att_g),
           'target_generated_grad_corr':correlation(grad_t,grad_g),
           'target_generated_grad_box_iou':best_box_iou(
               heatmap_boxes(grad_t,CFG.HEATMAP_QUANTILE),
               heatmap_boxes(grad_g,CFG.HEATMAP_QUANTILE))})
    return pd.DataFrame(out)

MATCHED_COMPARISONS=compare_target_generated_maps(MAP_ROWS)
if not MATCHED_COMPARISONS.empty:
    MATCHED_COMPARISONS.to_csv(CFG.OUTPUT_DIR/'target_generated_distribution_comparison.csv',index=False)
    display(MATCHED_COMPARISONS.groupby(['model','concept']).agg(
        n=('sample_id','nunique'),
        median_entropy_shift=('delta_grad_entropy','median'),
        median_top10_shift=('delta_grad_top10_mass','median'),
        median_gradient_corr=('target_generated_grad_corr','median')).round(4))
    caption_rows=MATCHED_COMPARISONS[MATCHED_COMPARISONS.concept=='caption']
    if not caption_rows.empty:
        fig,ax=plt.subplots(figsize=(8,4))
        models=sorted(caption_rows.model.unique())
        plots=[caption_rows[caption_rows.model==m]['delta_grad_top10_mass'].dropna().values
               for m in models]
        ax.boxplot(plots,tick_labels=models,showmeans=True)
        ax.axhline(0,linewidth=1,linestyle='--',color='gray')
        ax.set(ylabel='Change in top-10% gradient mass (generated − target)',
               title='Concentration shift for original target-caption gradients')
        plt.xticks(rotation=20);plt.tight_layout();plt.show()

In [ ]:
# 22 — Token and concept importance tables (no color calibration across models)

def report_importance(sample_id=None,top=12):
    sid=sample_id or PAIRS[0]['sample_id']
    if TOKEN_ROWS:
        frame=pd.DataFrame(TOKEN_ROWS)
        frame=frame[(frame.sample_id==sid)&(frame.prompt=='caption')&(frame.role=='target')]
        print('TOP TOKEN GRADIENT (within each CLIP model)')
        for model,group in frame.groupby('model'):
            display(group.nlargest(top,'grad_norm')[['model','position','token','grad_norm','text_self_attention']])
    if LOO_ROWS:
        frame=pd.DataFrame(LOO_ROWS)
        frame=frame[(frame.sample_id==sid)&(frame.role=='target')]
        print('LOO concept impact: cosine(full caption)-cosine(without concept)')
        display(frame.sort_values(['model','loo_delta'],ascending=[True,False]))

report_importance()

In [ ]:
# 23 — Consolidate across 10 pairs; export complete diagnostics

def export_all():
    manifest={'schema':'analysis-diffusion-concept-distribution-v1',
              'configuration':{k:str(v) if isinstance(v,Path) else v for k,v in vars(CFG).items()},
              'devices':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
              'source_captions_are_frozen':True,
              'note':'Not ASR; attention/gradient comparisons are observational diagnostics'}
    save_json(CFG.OUTPUT_DIR/'run_manifest.json',manifest)
    save_json(CFG.OUTPUT_DIR/'run_status.json',RUN_STATUS)
    save_json(ARTIFACTS/'map_records.json',MAP_ROWS)
    if QWEN_TOKEN_ROWS:
        pd.DataFrame(QWEN_TOKEN_ROWS).to_csv(CFG.OUTPUT_DIR/'qwen_text_token_attention.csv',index=False)
    for name,rows in [('map_summary.csv',MAP_ROWS),('clip_token_gradient.csv',TOKEN_ROWS),
                      ('clip_concept_loo.csv',LOO_ROWS),('model_status.csv',RUN_STATUS)]:
        if rows:pd.DataFrame(rows).to_csv(CFG.OUTPUT_DIR/name,index=False)
    table=pd.DataFrame(MAP_ROWS)
    if not table.empty:
        display(table.groupby(['model','role']).agg(
            n=('sample_id','nunique'),
            entropy=('grad_entropy','mean'),
            concentration=('grad_top10_mass','mean'),
            attn_grad_corr=('attn_grad_spearman','mean'),
            box_iou=('box_iou','mean')).round(4))
    display(pd.DataFrame(RUN_STATUS))
    print('Saved to:',CFG.OUTPUT_DIR)
    return table

SUMMARY = export_all()

In [ ]:
# 24 — Local CPU-only invariants, including gradient geometry, pairing and cache integrity

def run_small_tests():
    x=np.zeros((32,32),dtype=np.float32);x[4:13,5:12]=1
    assert len(heatmap_boxes(x,0.5))>=1
    assert box_iou([0,0,10,10],[5,5,15,15])==25/175
    assert abs(normalized_entropy(np.ones((16,16)))-1)<1e-5
    assert 0.0<top_mass(np.ones((10,10)))<0.2
    assert literal_concept_check('red dog','A red dog runs')=='red dog'
    assert literal_concept_check('blue dog','A red dog runs') is None
    assert phrase_removed('A red dog and cat','red dog')=='A and cat'
    assert truncate_words('a b c d',2)=='a b'
    assert len({p['sample_id'] for p in PAIRS})==len(PAIRS)
    assert len(PAIRS)==CFG.N_SAMPLES
    assert all(Path(x['source']).is_file() and Path(x['target']).is_file() for x in PAIRS)
    for rec in DIFFUSION_RESULTS.values():
        if rec['status']=='ok':assert sha256(rec['image'])==rec['image_sha256']
    for row in MAP_ROWS:
        with np.load(row['map_file']) as maps:
            for key in ('attention','gradient'):
                assert maps[key].shape==(224,224) and np.isfinite(maps[key]).all()
    print('CPU invariant tests passed; NOTE: this does NOT prove remote HF model execution.')

run_small_tests()

### Suggested experiment order

1. **Smoke test:** Set `N_SAMPLES=1`, `RUN_BLIP2=False`, `RUN_QWEN=False`, and one caption budget. Verify pair IDs, caption, generator and CLIP heatmaps.
2. **Ten-pair CLIP baseline:** `N_SAMPLES=10`, all three CLIP models; compare target vs generated concept saliency and LOO concept deltas.
3. **Diffusion ablation:** Fix source/caption/seed/steps, vary `strength=(0.45,0.7,0.9)` and caption budget `(12,24,40)`. Note that BLIP may generate captions shorter than all budgets (switch `CAPTION_BACKEND="qwen3vl"` to try longer image-grounded captions); inspect **actual word and diffusion token counts**.
4. **Expensive VLM pass:** Sequential `RUN_BLIP2=True` and `RUN_QWEN=True`; use 4-bit, two T4 GPUs, save after each pair. Failures are reported in `model_status.csv` rather than implicitly filled.
5. **Interpretation:** Compare **target** and **generated** concept gradients, concentration, and LOO cosine difference. Higher attention–gradient agreement alone **does not** establish a transferable adversarial attack advantage. A held-out transfer-ASR experiment would be needed for that claim.

**Limitations:** Real model forward/grad paths, Kaggle GPU memory and model checkpoint accessibility **must be verified on Kaggle**. No Kaggle execution is claimed by this notebook generator. BLIP-2 and Qwen3-VL gradient internals can change between `transformers` versions; their errors are intentionally exposed. `clip_g14` names a **ViT-g** checkpoint, not the substantially larger ViT-bigG. The final VLM model row counts are determined by successfully completed runs, not automatically assumed to be 10.